# 01 — Exploratory Data Analysis

Emotion classification on the Hugging Face [`emotion`](https://huggingface.co/datasets/dair-ai/emotion) dataset.
Each example is a short English text labelled with one of six emotions: **sadness, joy, love, anger, fear, surprise**.

This notebook only *explores* the data. Tokenization, feature extraction and modelling live in `02_Preprocessing.ipynb` and `03_Modelling.ipynb`.

## 1. Setup

In [ ]:
# Run once if the packages are missing (e.g. in Colab):
# !pip install -q -r ../requirements.txt

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

## 2. Load the dataset

In [ ]:
emotions = load_dataset("emotion", trust_remote_code=True)
emotions

The dataset ships with three predefined splits: **train** (16,000), **validation** (2,000) and **test** (2,000).
We keep these splits fixed throughout the project so that the test set stays untouched until the final evaluation.

In [ ]:
# Integer label -> human-readable emotion name. All splits share the same
# label feature, so a single mapping is enough.
label_names = emotions["train"].features["label"].names
int2str = emotions["train"].features["label"].int2str
label_names

## 3. Convert to pandas for exploration

In [ ]:
emotions.set_format(type="pandas")

df_train = emotions["train"][:]
df_val = emotions["validation"][:]
df_test = emotions["test"][:]

for df in (df_train, df_val, df_test):
    df["emotion_name"] = df["label"].apply(int2str)

df_train.head()

## 4. Class distribution

How balanced is the dataset? We look at each split separately to check that
train, validation and test follow a similar distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)

for ax, (name, df) in zip(
    axes, [("Training", df_train), ("Validation", df_val), ("Test", df_test)]
):
    df["emotion_name"].value_counts(ascending=True).plot.barh(ax=ax)
    ax.set_title(f"{name} data")
    ax.set_ylabel("Emotion")
    ax.set_xlabel("Count")

fig.suptitle("Frequency of classes per split", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

The dataset is clearly **imbalanced**: *joy* and *sadness* dominate while *surprise*
and *love* are rare. The three splits share the same shape, which is what we want.
We keep this imbalance in mind when reading the evaluation metrics in `03_Modelling.ipynb`
(hence we report a weighted F1 score, not only accuracy).

## 5. Text length

How long are the texts? This informs the `max_length` we choose for tokenization
in `02_Preprocessing.ipynb`.

In [ ]:
df_train["words_per_text"] = df_train["text"].str.split().apply(len)
df_train["words_per_text"].describe()

In [ ]:
df_train.boxplot(
    "words_per_text", by="emotion_name", grid=False, showfliers=False, color="blue"
)
plt.suptitle("")
plt.title("Words per text by emotion (training set)")
plt.xlabel("Emotion")
plt.ylabel("Words per text")
plt.show()

Most texts are well under ~50 words, so a tokenizer `max_length` of a few dozen
tokens covers the vast majority of examples without truncating meaningful content.

In [ ]:
# Restore the default (Arrow) format so later notebooks load a clean dataset.
emotions.reset_format()